# Module 6 - Lab 1: Evaluate Measurements in Jupyter

In this lab you use Jupyter as an analysis and documentation environment. You will load measurement data, inspect data quality, choose analysis parameters, create visualizations, and compare how different choices influence the result.

The goal is not only to run code. The goal is to make your assumptions visible:

- Which dataset did you use?
- Which columns did you analyse?
- Which time range, smoothing, and outlier settings did you choose?
- How do data quality and parameter choices change the interpretation?
- Which findings are exploratory, and which would you communicate as results?

## Learning goals
- Run and adapt a prepared analysis notebook in JupyterHub.
- Import measurement data from CSV or Excel files.
- Inspect structure, units, missing values, duplicates, and time steps.
- Use parameters to control filtering, smoothing, and outlier detection.
- Generate visualizations and simple summary statistics.
- Document observations directly in the notebook.
- Prepare analysis results so another group can understand and reuse them.

## Section 1: Notebook Orientation

A Jupyter notebook combines text cells and code cells.

- Text cells explain assumptions, decisions, observations, and conclusions.
- Code cells load data, transform it, create plots, and calculate results.
- Running cells in order matters because later cells often depend on variables created earlier.

Before changing the notebook, run all cells once with the example dataset. Then adapt one parameter at a time and observe what changes.

In [ ]:
# Section 1: Import Required Libraries
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from metadata_loader import (
    apply_recorded_data_path_override,
    load_metadata_context,
    save_public_metadata,
    summarize_metadata_context,
)
from data_format_loader import load_recorded_data, summarize_loaded_data

pd.set_option('display.max_columns', 30)
pd.set_option('display.precision', 4)

print('Libraries imported successfully.')

## Section 2: Load Metadata and Recorded Data Metadata

`metadata.json` records the selected data file, measurement type (`drivetrain` or `suspension`), run name, quantity, data stage, version, and optional hot-storage path. The detailed recording metadata is still extracted from the data file or its adjacent `meta/` folder.

In [ ]:
# Section 2: Load Metadata and Recorded Data Metadata
metadata_context = load_metadata_context(project_root)
metadata = metadata_context['public_metadata']
metadata_path = metadata_context['public_metadata_path']
selected_data_path = metadata_context['selected_data_path']

loaded_recorded_data = load_recorded_data(selected_data_path, project_root)
recorded_data_metadata = {
    'recorded_data_path': loaded_recorded_data['path'],
    'detected_format': loaded_recorded_data['format'],
    'extracted_metadata': loaded_recorded_data['recording_metadata'],
}

print('Metadata context:')
display(pd.json_normalize(summarize_metadata_context(metadata_context), sep='.').T.rename(columns={0: 'value'}))

print('Recorded data metadata:')
display(pd.json_normalize(recorded_data_metadata, sep='.').T.rename(columns={0: 'value'}))

### Optional: Override Recorded Data Path if Needed

Use this cell if you need to analyse another group's recorded data. The override changes the pointer to the data file; extracted recording metadata is then reloaded from that file or its `meta/` folder.

In [ ]:
# Section 3: Override Recorded Data Path if Needed
recorded_data_path_override = None
measurement_type_override = None
# recorded_data_path_override = 'data/suspension/Example/Beschleunigung ohne g.xls'
# measurement_type_override = 'suspension'
# recorded_data_path_override = 'data/drivetrain/Example/Raw Data.csv'
# measurement_type_override = 'drivetrain'

save_path_override = False

metadata = apply_recorded_data_path_override(
    metadata,
    recorded_data_path_override=recorded_data_path_override,
    measurement_type_override=measurement_type_override,
)
selected_data_path = project_root / metadata['recorded_data_path']

if recorded_data_path_override:
    loaded_recorded_data = load_recorded_data(selected_data_path, project_root)
    recorded_data_metadata = {
        'recorded_data_path': loaded_recorded_data['path'],
        'detected_format': loaded_recorded_data['format'],
        'extracted_metadata': loaded_recorded_data['recording_metadata'],
    }

if save_path_override:
    save_public_metadata(metadata, project_root)
    print('Saved recorded data path override:', metadata_path)
else:
    print('Override applied for this notebook session only.' if recorded_data_path_override else 'No override applied.')

print('Recorded data:', selected_data_path)
print('Measurement type:', metadata.get('measurement_type'))
print('Detected format:', recorded_data_metadata['detected_format']['format_label'])
display(pd.json_normalize(recorded_data_metadata, sep='.').T.rename(columns={0: 'value'}))

## Section 4: Select a Dataset

The selected dataset comes from `metadata.json` unless you used the override cell above.

In [ ]:
# Section 4: Select a Dataset
data_root = project_root / 'data'

available_files = sorted(
    [p for p in data_root.rglob('*') if p.suffix.lower() in ['.csv', '.xlsx', '.xls']]
)

print('Available measurement files:')
for i, path in enumerate(available_files):
    print(f'{i}: {path.relative_to(project_root)}')

print('\nSelected dataset from metadata pointer:')
print(selected_data_path)
print('Detected format:', recorded_data_metadata['detected_format']['format_label'])

## Section 5: Load the Measurement Data

This cell reads the selected file into a table called `df_raw`.

After loading, inspect the first rows. Check whether the column names, units, and values match what you expected from the measurement setup.

In [ ]:
# Section 5: Load the Measurement Data
loaded_recorded_data = load_recorded_data(selected_data_path, project_root)
df_raw = loaded_recorded_data['table']

recorded_data_metadata = {
    'recorded_data_path': loaded_recorded_data['path'],
    'detected_format': loaded_recorded_data['format'],
    'extracted_metadata': loaded_recorded_data['recording_metadata'],
}

print('Loaded file:', selected_data_path.name)
print('Detected format:', recorded_data_metadata['detected_format']['format_label'])
print('Rows, columns:', df_raw.shape)
display(df_raw.head())
display(pd.DataFrame([summarize_loaded_data(loaded_recorded_data)]).T.rename(columns={0: 'value'}))

## Observation 1: First Impression

Write down your first observations before changing any code.

- What was measured?
- Which columns contain time and measurement values?
- Are the units clear?
- Do the first rows look plausible?

**Your notes:**

- 

## Section 6: Inspect Data Structure and Quality

Before evaluating results, check whether the data can be trusted.

Typical questions:

- Are values missing?
- Are rows duplicated?
- Are numeric columns really numeric?
- Does the time column increase steadily?
- Are there unusual jumps or values outside a plausible range?

In [ ]:
# Section 6: Inspect Data Structure and Quality
print('Column names:')
print(df_raw.columns.tolist())

print('\nData types:')
print(df_raw.dtypes)

print('\nMissing values per column:')
display(df_raw.isna().sum().to_frame('missing_values'))

print('Duplicate rows:', df_raw.duplicated().sum())

print('\nSummary statistics:')
display(df_raw.describe(include='all').T)

## Section 7: Define Analysis Parameters

Parameters are explicit analysis choices. Change them and rerun the following cells to see how the output changes.

Important: A parameter is not automatically right because it produces a clean plot. It must fit the measurement question and the data quality.

In [ ]:
# Section 7: Define Analysis Parameters
recorded_columns = []
try:
    recorded_columns = df_raw.columns.tolist()
except NameError:
    pass

# Set these manually if automatic column detection does not choose the right columns.
time_column = next((c for c in recorded_columns if 'time' in c.lower() or 'zeit' in c.lower()), 'Time (s)')
value_column = None

# Optional time filter. Use None to keep the full measurement.
analysis_start_s = None
analysis_end_s = None

# Smoothing and outlier parameters.
smoothing_window = 5
outlier_z_threshold = 3.0

# Plot settings.
plot_raw_values = True
plot_smoothed_values = True

print('Parameters set.')
print('Time column:', time_column)
print('Value column:', value_column)
print('Smoothing window:', smoothing_window)
print('Outlier z-threshold:', outlier_z_threshold)

In [ ]:
# Section 7: Prepare Columns for Analysis
df = df_raw.copy()

# Convert columns to numeric where possible. Non-numeric text becomes NaN.
for col in df.columns:
    converted = pd.to_numeric(df[col], errors='coerce')
    if converted.notna().sum() > 0:
        df[col] = converted

numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

if time_column not in df.columns:
    time_candidates = [c for c in numeric_columns if 'time' in c.lower() or 'zeit' in c.lower()]
    time_column = time_candidates[0] if time_candidates else numeric_columns[0]

if value_column is None:
    value_candidates = [c for c in numeric_columns if c != time_column]
    value_column = value_candidates[0]

df_analysis = df[[time_column, value_column]].dropna().copy()
df_analysis = df_analysis.sort_values(time_column)

if analysis_start_s is not None:
    df_analysis = df_analysis[df_analysis[time_column] >= analysis_start_s]
if analysis_end_s is not None:
    df_analysis = df_analysis[df_analysis[time_column] <= analysis_end_s]

df_analysis = df_analysis.reset_index(drop=True)

print('Time column:', time_column)
print('Value column:', value_column)
print('Rows used for analysis:', len(df_analysis))
display(df_analysis.head())

## Section 8: Time Step and Data Quality Checks

Many measurements depend on time. If the time steps are irregular, averages, slopes, and plots can become misleading.

Use this section to check whether the measurement frequency is stable enough for your question.

In [ ]:
# Section 8: Time Step and Data Quality Checks
time_diff = df_analysis[time_column].diff().dropna()

quality_report = pd.DataFrame({
    'metric': [
        'rows_used',
        'time_min',
        'time_max',
        'duration',
        'median_time_step',
        'min_time_step',
        'max_time_step',
        'non_increasing_time_steps'
    ],
    'value': [
        len(df_analysis),
        df_analysis[time_column].min(),
        df_analysis[time_column].max(),
        df_analysis[time_column].max() - df_analysis[time_column].min(),
        time_diff.median() if len(time_diff) else np.nan,
        time_diff.min() if len(time_diff) else np.nan,
        time_diff.max() if len(time_diff) else np.nan,
        int((time_diff <= 0).sum()) if len(time_diff) else 0
    ]
})

display(quality_report)

plt.figure(figsize=(8, 3))
plt.plot(time_diff.values, marker='.', linestyle='none')
plt.axhline(time_diff.median(), color='orange', label='median time step')
plt.title('Time Step Check')
plt.xlabel('Row interval')
plt.ylabel('Time difference')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Observation 2: Data Quality

Record what might influence the analysis.

- Are there missing values or duplicates?
- Are the time steps regular enough?
- Are there parts of the measurement that should be excluded?
- Which quality issues are minor, and which could change the result?

**Your notes:**

- 

## Section 9: Visualize Raw and Smoothed Data

Smoothing can make trends easier to see, but it can also hide short events. Compare the raw values with the smoothed curve and decide whether smoothing is appropriate for your measurement question.

In [ ]:
# Section 9: Visualize Raw and Smoothed Data
df_analysis['smoothed'] = df_analysis[value_column].rolling(
    window=smoothing_window,
    center=True,
    min_periods=1
).mean()

plt.figure(figsize=(10, 4))

if plot_raw_values:
    plt.plot(
        df_analysis[time_column],
        df_analysis[value_column],
        label='raw values',
        alpha=0.45
    )

if plot_smoothed_values:
    plt.plot(
        df_analysis[time_column],
        df_analysis['smoothed'],
        label=f'smoothed, window={smoothing_window}',
        linewidth=2
    )

plt.title('Measurement Values Over Time')
plt.xlabel(time_column)
plt.ylabel(value_column)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Section 10: Detect Possible Outliers

This simple check marks values whose distance from the mean is larger than `outlier_z_threshold` standard deviations.

This is only a first signal. A marked value is not automatically wrong. It might be a real event, a sensor problem, or an artefact of the chosen threshold.

In [ ]:
# Section 10: Detect Possible Outliers
value_mean = df_analysis[value_column].mean()
value_std = df_analysis[value_column].std(ddof=0)

if value_std == 0 or np.isnan(value_std):
    df_analysis['z_score'] = 0.0
else:
    df_analysis['z_score'] = (df_analysis[value_column] - value_mean) / value_std

df_analysis['possible_outlier'] = df_analysis['z_score'].abs() > outlier_z_threshold

print('Possible outliers:', int(df_analysis['possible_outlier'].sum()))
display(df_analysis[df_analysis['possible_outlier']].head(10))

plt.figure(figsize=(10, 4))
plt.plot(df_analysis[time_column], df_analysis[value_column], label='raw values', alpha=0.45)
plt.scatter(
    df_analysis.loc[df_analysis['possible_outlier'], time_column],
    df_analysis.loc[df_analysis['possible_outlier'], value_column],
    color='red',
    label='possible outlier',
    zorder=3
)
plt.title('Possible Outliers')
plt.xlabel(time_column)
plt.ylabel(value_column)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Section 11: Compare Parameter Choices

Now compare multiple smoothing windows. This shows how a code choice can change the visible result.

Try at least three settings. Use one small value, one medium value, and one large value.

In [ ]:
# Section 11: Compare Parameter Choices
smoothing_windows_to_compare = [1, 5, 15, 31]

comparison = df_analysis[[time_column, value_column]].copy()

plt.figure(figsize=(10, 4))
plt.plot(
    comparison[time_column],
    comparison[value_column],
    color='lightgray',
    label='raw values'
)

summary_rows = []
for window in smoothing_windows_to_compare:
    column_name = f'smoothed_{window}'
    comparison[column_name] = comparison[value_column].rolling(
        window=window,
        center=True,
        min_periods=1
    ).mean()
    plt.plot(comparison[time_column], comparison[column_name], label=f'window={window}')
    summary_rows.append({
        'smoothing_window': window,
        'min': comparison[column_name].min(),
        'max': comparison[column_name].max(),
        'mean': comparison[column_name].mean(),
        'standard_deviation': comparison[column_name].std()
    })

plt.title('Comparison of Smoothing Parameters')
plt.xlabel(time_column)
plt.ylabel(value_column)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

comparison_summary = pd.DataFrame(summary_rows)
display(comparison_summary)

## Observation 3: Parameter Effects

Compare the outputs from the previous section.

- Which smoothing window preserves short events?
- Which smoothing window makes the overall trend clearest?
- Which setting would you use for exploration?
- Which setting would you communicate in a result, and why?

**Your notes:**

- 

## Section 12: Simple Evaluation Summary

This section creates a compact result table from the selected parameters, the simple course metadata, the detected file format, and the metadata source.

In [ ]:
# Section 12: Simple Evaluation Summary
result_summary = pd.DataFrame({
    'item': [
        'dataset',
        'measurement_type',
        'run_name',
        'quantity',
        'data_stage',
        'version',
        'detected_format',
        'metadata_source',
        'time_column',
        'value_column',
        'rows_used',
        'analysis_start',
        'analysis_end',
        'smoothing_window',
        'outlier_z_threshold',
        'possible_outliers',
        'minimum_value',
        'maximum_value',
        'mean_value',
        'median_value',
        'standard_deviation'
    ],
    'value': [
        str(selected_data_path.relative_to(project_root)),
        metadata.get('measurement_type'),
        metadata.get('run_name'),
        metadata.get('quantity'),
        metadata.get('data_stage'),
        metadata.get('version'),
        recorded_data_metadata['detected_format']['format_label'],
        recorded_data_metadata['extracted_metadata'].get('source'),
        time_column,
        value_column,
        len(df_analysis),
        df_analysis[time_column].min(),
        df_analysis[time_column].max(),
        smoothing_window,
        outlier_z_threshold,
        int(df_analysis['possible_outlier'].sum()),
        df_analysis[value_column].min(),
        df_analysis[value_column].max(),
        df_analysis[value_column].mean(),
        df_analysis[value_column].median(),
        df_analysis[value_column].std()
    ]
})

display(result_summary)

## Section 13: Communicate Exploratory Steps and Results

Use this final section to separate exploration from results.

### Exploratory steps

List the checks and parameter experiments that helped you understand the data.

- 

### Result you would communicate

Write a short result statement. Include the dataset, relevant parameters, and one sentence about data quality.

- 

### Open questions for reuse by another group

What should another group know before reusing your analysis?

- 